# BioMini Phase I Educational Materials

## Notebook 07 — Persistence, Provenance and Reproducibility
### Scientific Object를 저장하고, 추적하고, 다시 사용할 수 있게 만들기

Notebook 06에서는 Dataset으로 model을 학습하고 `ProteinPredictor`를 통해 prediction까지 수행했습니다.

하지만 Python session이 끝나면 object와 model은 사라집니다.
또 model만 저장한다고 해서 reproducible scientific workflow가 완성되는 것도 아닙니다.

이번 Notebook에서는 다음을 다룹니다.

```text
serialization
schema version
stable identity
metadata
provenance
lineage
ModelArtifact
joblib persistence
```

핵심 질문은 다음입니다.

> scientific result를 나중에 다시 사용하려면 무엇을 함께 보존해야 하는가?

---

## Learning Objectives

1. serialization과 persistence의 차이를 설명한다.
2. domain object를 dictionary/JSON으로 변환하고 복원한다.
3. schema version이 왜 필요한지 설명한다.
4. stable ID와 metadata의 의미를 이해한다.
5. provenance와 lineage를 설명한다.
6. DNA → RNA → Protein lineage를 추적한다.
7. nested object serialization에서 schema validation이 왜 필요한지 이해한다.
8. trained model과 inference context를 `ModelArtifact`로 함께 보존한다.
9. joblib save/load 후 prediction consistency를 확인한다.
10. pickle/joblib 기반 artifact의 security risk를 이해한다.

## 1. 왜 Persistence가 필요한가?

Scientific software에서는 다음 상태를 나중에 다시 사용해야 합니다.

```text
biological object
feature representation
dataset
trained model
model metadata
feature schema
provenance
```

단순히 Python object를 메모리에 가지고 있는 것만으로는 충분하지 않습니다.

```text
in-memory object
    ↓ save
persistent representation
    ↓ load
restored object
```

## 2. Serialization과 Persistence

두 개념을 구분합니다.

```text
Serialization
= object를 저장 가능한 data structure로 변환

Persistence
= 그 representation을 file 등에 실제로 저장하고 나중에 복원
```

예:

```text
Protein
 ↓ to_dict()
dictionary
 ↓ json.dump()
JSON file
```

## 3. Schema Version

software는 변하지만 저장된 data는 남습니다.

예를 들어 v1의 JSON과 v2의 object structure가 다를 수 있습니다.

따라서 serialized data에는 version information이 필요합니다.

In [ ]:
SCHEMA_VERSION = "1.0"


class SerializationError(Exception):
    pass


def validate_schema_version(data: dict) -> None:
    if not isinstance(data, dict):
        raise SerializationError(
            "serialized payload must be a dictionary"
        )

    schema_version = data.get(
        "schema_version"
    )

    if schema_version != SCHEMA_VERSION:
        raise SerializationError(
            f"unsupported schema_version: {schema_version!r}; "
            f"expected {SCHEMA_VERSION!r}"
        )

In [ ]:
good_payload = {
    "schema_version": "1.0"
}

validate_schema_version(
    good_payload
)

print("schema validation: PASS")

In [ ]:
bad_payload = {
    "schema_version": "99.0"
}

try:
    validate_schema_version(
        bad_payload
    )
except SerializationError as exc:
    print(
        type(exc).__name__,
        ":",
        exc
    )

## 4. Stable Identity

name만으로 object identity를 표현하면 충분하지 않을 수 있습니다.

예:

```text
Protein("sample")
Protein("sample")
```

두 object가 같은 이름을 가질 수 있습니다.

따라서 canonical BioMini에서는 UUID 기반 stable ID를 사용합니다.

In [ ]:
from uuid import uuid4


class DomainEntity:
    schema_version = SCHEMA_VERSION

    def __init__(
        self,
        name: str,
        entity_id: str | None = None,
        metadata: dict | None = None,
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "name must be a non-empty string"
            )

        self.id = (
            entity_id
            if entity_id is not None
            else str(uuid4())
        )

        self.name = name.strip()
        self.metadata = dict(
            metadata or {}
        )

In [ ]:
a = DomainEntity("sample")
b = DomainEntity("sample")

print("A:", a.id)
print("B:", b.id)

assert a.id != b.id

## 5. Metadata

scientific object에는 core scientific state와 별도로 contextual information이 필요할 수 있습니다.

예:

```text
organism
experiment
operator
batch
instrument
note
```

이런 정보는 object의 primary identity와 분리해 metadata로 관리할 수 있습니다.

In [ ]:
protein_metadata = {
    "organism": "Homo sapiens",
    "batch": "B-001",
    "note": "training example",
}

entity = DomainEntity(
    "protein_example",
    metadata=protein_metadata,
)

print(entity.metadata)

## 6. Provenance

Provenance는 object가 어디에서 왔는지를 기록합니다.

canonical BioMini의 minimal provenance는 다음 정보를 가집니다.

```text
source
source_id
parent_ids
metadata
```

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Provenance:
    source: str | None = None
    source_id: str | None = None
    parent_ids: list[str] = field(
        default_factory=list
    )
    metadata: dict = field(
        default_factory=dict
    )

    def to_dict(self) -> dict:
        return {
            "source": self.source,
            "source_id": self.source_id,
            "parent_ids": list(
                self.parent_ids
            ),
            "metadata": dict(
                self.metadata
            ),
        }

    @classmethod
    def from_dict(
        cls,
        data: dict | None
    ):
        if not data:
            return cls()

        return cls(
            source=data.get("source"),
            source_id=data.get("source_id"),
            parent_ids=list(
                data.get(
                    "parent_ids",
                    []
                )
            ),
            metadata=dict(
                data.get(
                    "metadata",
                    {}
                )
            ),
        )

## 7. DomainEntity에 Provenance 추가

In [ ]:
class DomainEntity:
    schema_version = SCHEMA_VERSION

    def __init__(
        self,
        name: str,
        entity_id: str | None = None,
        metadata: dict | None = None,
        provenance: Provenance | None = None,
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "name must be a non-empty string"
            )

        self.id = (
            entity_id
            if entity_id is not None
            else str(uuid4())
        )

        self.name = name.strip()
        self.metadata = dict(
            metadata or {}
        )

        self.provenance = (
            provenance
            if provenance is not None
            else Provenance()
        )

    def to_dict(self) -> dict:
        return {
            "schema_version":
                self.schema_version,

            "type":
                self.__class__.__name__,

            "id":
                self.id,

            "name":
                self.name,

            "metadata":
                dict(self.metadata),

            "provenance":
                self.provenance.to_dict(),
        }

## 8. Sequence Object Serialization

이제 sequence object도 `to_dict()` / `from_dict()`를 가질 수 있습니다.

In [ ]:
class SequenceMolecule(
    DomainEntity
):
    def __init__(
        self,
        name: str,
        sequence: str,
        entity_id: str | None = None,
        metadata: dict | None = None,
        provenance: Provenance | None = None,
    ):
        super().__init__(
            name=name,
            entity_id=entity_id,
            metadata=metadata,
            provenance=provenance,
        )

        self.sequence = (
            "".join(
                sequence.split()
            )
            .upper()
        )

    def to_dict(self) -> dict:
        data = super().to_dict()
        data["sequence"] = (
            self.sequence
        )
        return data

    @classmethod
    def from_dict(
        cls,
        data: dict
    ):
        validate_schema_version(
            data
        )

        return cls(
            name=data["name"],
            sequence=data["sequence"],
            entity_id=data.get("id"),
            metadata=data.get(
                "metadata"
            ),
            provenance=(
                Provenance.from_dict(
                    data.get(
                        "provenance"
                    )
                )
            ),
        )

In [ ]:
protein = SequenceMolecule(
    "protein1",
    "ACDE",
    metadata={
        "source_lab":
            "training_lab"
    }
)

payload = protein.to_dict()

payload

## 9. Object Restoration

serialized dictionary에서 object를 다시 복원할 수 있습니다.

In [ ]:
restored = (
    SequenceMolecule.from_dict(
        payload
    )
)

print(
    restored.name
)

print(
    restored.sequence
)

print(
    restored.id
)

assert (
    restored.id
    == protein.id
)

stable ID가 유지된다는 점이 중요합니다.

새 object를 만든 것이 아니라,
**같은 scientific entity를 복원한 것**으로 취급하기 때문입니다.

## 10. JSON Persistence

이제 dictionary를 JSON file로 저장합니다.

In [ ]:
import json
from pathlib import Path


def save_json(
    obj,
    path: str
):
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            obj.to_dict(),
            file,
            indent=2
        )

In [ ]:
json_path = (
    "protein_example.json"
)

save_json(
    protein,
    json_path
)

print(
    Path(json_path).read_text(
        encoding="utf-8"
    )
)

## 11. JSON Load

file에서 payload를 읽고 object를 복원합니다.

In [ ]:
def load_sequence_json(
    path: str
):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:
        data = json.load(
            file
        )

    validate_schema_version(
        data
    )

    return (
        SequenceMolecule.from_dict(
            data
        )
    )

In [ ]:
loaded = load_sequence_json(
    json_path
)

print(
    loaded.name,
    loaded.sequence,
    loaded.id
)

assert (
    loaded.id
    == protein.id
)

## 12. Lineage — DNA → RNA → Protein

Provenance의 가장 중요한 용도 중 하나는 derived object의 lineage를 추적하는 것입니다.

예:

```text
DNA
id = D1
 ↓ transcription

RNA
id = R1
parent_ids = [D1]
 ↓ translation

Protein
id = P1
parent_ids = [R1]
```

In [ ]:
class DNA(
    SequenceMolecule
):
    def transcribe(self):
        return RNA(
            self.name + "_RNA",
            self.sequence.replace(
                "T",
                "U"
            ),
            metadata=self.metadata,
            provenance=Provenance(
                source="transcription",
                parent_ids=[
                    self.id
                ],
            ),
        )


class RNA(
    SequenceMolecule
):
    codon_table = {
        "AUG": "M",
        "GCC": "A",
        "UUU": "F",
        "UAA": "*",
    }

    def translate(self):
        amino_acids = []

        for i in range(
            0,
            len(self.sequence) - 2,
            3
        ):
            codon = (
                self.sequence[
                    i:i + 3
                ]
            )

            aa = (
                self.codon_table[
                    codon
                ]
            )

            if aa == "*":
                break

            amino_acids.append(
                aa
            )

        return Protein(
            self.name + "_protein",
            "".join(
                amino_acids
            ),
            metadata=self.metadata,
            provenance=Provenance(
                source="translation",
                parent_ids=[
                    self.id
                ],
            ),
        )


class Protein(
    SequenceMolecule
):
    pass

In [ ]:
dna = DNA(
    "gene1",
    "ATGGCCTTT"
)

rna = dna.transcribe()
protein = rna.translate()

print(
    "DNA ID:",
    dna.id
)

print(
    "RNA parent:",
    rna.provenance.parent_ids
)

print(
    "Protein parent:",
    protein.provenance.parent_ids
)

assert (
    rna.provenance.parent_ids
    == [dna.id]
)

assert (
    protein.provenance.parent_ids
    == [rna.id]
)

이제 object graph는 단순 값의 집합이 아니라
**scientific derivation history**를 가집니다.

## 13. 왜 Provenance가 중요한가?

Scientific workflow에서는 다음 질문에 답할 수 있어야 합니다.

```text
이 Protein은 어떤 RNA에서 왔는가?
이 RNA는 어떤 DNA에서 왔는가?
어떤 experiment 또는 source에서 생성되었는가?
어떤 metadata가 함께 있었는가?
```

이 정보는 다음 단계에서 더 중요해집니다.

```text
Scientific AI
Self-Driving Lab
multi-step experiment
automated decision loop
```

즉 provenance는 단순 metadata decoration이 아니라
**traceability infrastructure**입니다.

## 14. Nested Serialization

`FeatureSet`, `Sample`, `Dataset`처럼 object 안에 다른 object가 들어가는 경우를 생각해 봅니다.

```text
Dataset
 └── Sample
      └── FeatureSet
```

각 layer가 serialization responsibility를 가져야 합니다.

In [ ]:
class FeatureSet:
    schema_version = SCHEMA_VERSION

    def __init__(
        self,
        name: str,
        features: dict
    ):
        self.name = name
        self.features = dict(
            features
        )

    def to_dict(self):
        return {
            "schema_version":
                self.schema_version,
            "type":
                "FeatureSet",
            "name":
                self.name,
            "features":
                dict(self.features),
        }

    @classmethod
    def from_dict(
        cls,
        data
    ):
        validate_schema_version(
            data
        )

        return cls(
            data["name"],
            data["features"],
        )


class Sample:
    schema_version = SCHEMA_VERSION

    def __init__(
        self,
        feature_set,
        target=None
    ):
        self.feature_set = (
            feature_set
        )
        self.target = target

    def to_dict(self):
        return {
            "schema_version":
                self.schema_version,
            "type":
                "Sample",
            "feature_set":
                self.feature_set.to_dict(),
            "target":
                self.target,
        }

    @classmethod
    def from_dict(
        cls,
        data
    ):
        validate_schema_version(
            data
        )

        return cls(
            FeatureSet.from_dict(
                data[
                    "feature_set"
                ]
            ),
            target=data.get(
                "target"
            ),
        )

## 15. Nested Schema Validation

중요한 edge case:

top-level `Sample`의 schema version은 올바르지만
내부 `FeatureSet`의 schema version이 잘못되어 있을 수 있습니다.

따라서 nested object도 **자신의 schema를 직접 검증**해야 합니다.

In [ ]:
bad_nested_payload = {
    "schema_version": "1.0",
    "type": "Sample",
    "feature_set": {
        "schema_version": "99.0",
        "type": "FeatureSet",
        "name": "bad",
        "features": {
            "length": 10.0
        },
    },
    "target": 1.0,
}

try:
    Sample.from_dict(
        bad_nested_payload
    )

except SerializationError as exc:
    print(
        type(exc).__name__,
        ":",
        exc
    )

이것이 Phase I Canonical Audit에서 실제로 수정된 중요한 문제 중 하나입니다.

핵심 원칙:

> top-level object만 schema validation하는 것으로는 충분하지 않다.

각 serializable object는 자신의 schema version을 검증해야 합니다.

## 16. Model Persistence의 문제

Notebook 06에서 trained model을 만들었습니다.

model만 저장하면 충분할까요?

아닙니다.

inference에는 최소한 다음 context가 필요합니다.

```text
trained model
feature columns
model name
target name
metadata
schema version
```

## 17. ModelArtifact

canonical BioMini는 이 context를 `ModelArtifact`로 묶습니다.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone


@dataclass
class ModelArtifact:
    model: object
    feature_columns: list[str]
    model_name: str
    target_name: str
    metadata: dict = field(
        default_factory=dict
    )
    schema_version: str = (
        SCHEMA_VERSION
    )
    created_at: str = field(
        default_factory=lambda:
            datetime.now(
                timezone.utc
            ).isoformat()
    )

`ModelArtifact`의 핵심은 model을 감싸는 것 자체가 아닙니다.

다음 질문에 답할 수 있게 하는 것입니다.

```text
어떤 model인가?
어떤 feature schema를 사용했는가?
무엇을 예측하는가?
언제 생성되었는가?
어떤 metadata가 있는가?
```

## 18. 실제 scikit-learn Model 생성

간단한 training data를 만듭니다.

In [ ]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


X = pd.DataFrame({
    "length": [
        5.0,
        10.0,
        15.0,
        20.0,
    ],
    "hydrophobic_fraction": [
        0.2,
        0.4,
        0.6,
        0.8,
    ],
})

y = pd.Series([
    0.1,
    0.3,
    0.6,
    0.9,
])

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(alpha=1.0)
    ),
])

model.fit(
    X,
    y
)

print(
    model.predict(
        X
    )
)

## 19. ModelArtifact 생성

In [ ]:
artifact = ModelArtifact(
    model=model,
    feature_columns=list(
        X.columns
    ),
    model_name="protein_ridge",
    target_name="synthetic_activity",
    metadata={
        "purpose":
            "Notebook 07 example"
    },
)

print(
    artifact.model_name
)

print(
    artifact.feature_columns
)

print(
    artifact.created_at
)

## 20. joblib로 ModelArtifact 저장

scikit-learn object는 JSON으로 직접 저장하기 어렵습니다.

Phase I에서는 `joblib`을 사용합니다.

In [ ]:
import joblib


artifact_path = (
    "protein_model.joblib"
)

joblib.dump(
    artifact,
    artifact_path
)

print(
    artifact_path
)

## 21. ModelArtifact Load

In [ ]:
loaded_artifact = joblib.load(
    artifact_path
)

print(
    loaded_artifact.model_name
)

print(
    loaded_artifact.feature_columns
)

## 22. Prediction Consistency

save/load 전후 prediction이 같은지 확인합니다.

In [ ]:
X_test = pd.DataFrame({
    "length": [
        12.0
    ],
    "hydrophobic_fraction": [
        0.5
    ],
})

before = artifact.model.predict(
    X_test[
        artifact.feature_columns
    ]
)[0]

after = (
    loaded_artifact.model.predict(
        X_test[
            loaded_artifact.feature_columns
        ]
    )[0]
)

print(
    "before:",
    before
)

print(
    "after :",
    after
)

assert abs(
    before - after
) < 1e-12

이 검증은 중요합니다.

단순히 file이 저장되었다는 것만 확인하는 것이 아니라:

```text
save
 ↓
load
 ↓
same inference behavior
```

를 확인하기 때문입니다.

## 23. ProteinPredictor와 ModelArtifact

canonical BioMini에서는 `ProteinPredictor.from_artifact()`를 제공합니다.

개념:

```text
ModelArtifact
 ├── trained model
 └── feature columns
        ↓
ProteinPredictor
```

즉 artifact를 load하면 predictor를 다시 구성할 수 있습니다.

In [ ]:
class ProteinPredictor:
    def __init__(
        self,
        model,
        feature_columns
    ):
        self.model = model
        self.feature_columns = list(
            feature_columns
        )

    @classmethod
    def from_artifact(
        cls,
        artifact
    ):
        return cls(
            artifact.model,
            artifact.feature_columns
        )

In [ ]:
predictor = (
    ProteinPredictor
    .from_artifact(
        loaded_artifact
    )
)

print(
    predictor.feature_columns
)

## 24. Security Warning — joblib과 pickle

매우 중요한 주의점입니다.

`joblib`은 pickle 기반입니다.

pickle 기반 file은 load 과정에서 arbitrary Python code execution 위험이 있을 수 있습니다.

따라서:

> **신뢰할 수 없는 `.joblib`, `.pkl` file을 load하면 안 됩니다.**

Phase I canonical security policy도 동일합니다.

```text
load model artifacts only from trusted sources
```

## 25. JSON과 joblib의 역할 차이

BioMini Phase I에서는 대략 다음처럼 나눕니다.

```text
Domain / Data Object
    ↓
JSON

Trained ML Model
    ↓
ModelArtifact
    ↓
joblib
```

이렇게 사용하는 이유는 representation의 특성이 다르기 때문입니다.

JSON:

```text
human-readable
language-neutral에 가까움
simple structured data에 적합
```

joblib:

```text
Python/scikit-learn object persistence에 편리
하지만 trusted-source requirement가 큼
```

## 26. Self-Check

In [ ]:
entity = DomainEntity(
    "self_check"
)

payload = entity.to_dict()

assert (
    payload["schema_version"]
    == "1.0"
)

assert (
    payload["id"]
    == entity.id
)

dna = DNA(
    "gene",
    "ATGGCCTTT"
)

rna = dna.transcribe()
protein = rna.translate()

assert (
    rna.provenance.parent_ids
    == [dna.id]
)

assert (
    protein.provenance.parent_ids
    == [rna.id]
)

assert (
    loaded_artifact.feature_columns
    == list(X.columns)
)

assert abs(
    before - after
) < 1e-12

print(
    "Notebook 07 self-check: PASS"
)

## 27. Canonical BioMini와 비교

실제 BioMini package가 설치되어 있다면 canonical persistence layer를 사용할 수 있습니다.

In [ ]:
try:
    from biomini import (
        DNA as BioMiniDNA,
        Protein as BioMiniProtein,
        Provenance as BioMiniProvenance,
        ModelArtifact as BioMiniModelArtifact,
        save_json as biomini_save_json,
        load_json as biomini_load_json,
        save_model_artifact,
        load_model_artifact,
        ProteinPredictor as BioMiniProteinPredictor,
    )

    bm_dna = BioMiniDNA(
        "canonical_gene",
        "ATGGCCTTT"
    )

    biomini_save_json(
        bm_dna,
        "canonical_dna.json"
    )

    restored_bm_dna = (
        biomini_load_json(
            "canonical_dna.json"
        )
    )

    print(
        restored_bm_dna
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

## 28. Reproducibility를 위한 최소 Context

이 Notebook의 핵심을 한 문장으로 정리하면:

> scientific result를 재현하려면 값만 저장하는 것이 아니라,
> 그 값을 만든 context와 lineage도 함께 보존해야 한다.

따라서 BioMini는 다음을 분리해서 관리합니다.

```text
identity
metadata
provenance
schema
features
model
feature columns
target information
```

## 29. Bridge to Notebook 08

이제 BioMini에는 다음이 있습니다.

```text
domain model
analysis
feature engineering
dataset
machine learning
serialization
model persistence
provenance
```

하지만 아직 마지막 질문이 남습니다.

> 이 software가 정말 의도대로 작동하는가?

그리고 더 중요한 질문:

> scientific behavior가 independent reference와도 일치하는가?

Notebook 08에서는 다음을 다룹니다.

```text
pytest
regression testing
scientific reference validation
Biopython comparison
package structure
pyproject.toml
version management
dependency management
GitHub Actions CI
```

다음 주제:

> **Validation, Packaging and Scientific Software**

## 30. Exercises

### Exercise 1 — JSON Round Trip
`SequenceMolecule`을 JSON으로 저장한 후 다시 load하고
name, sequence, ID가 같은지 확인하십시오.

### Exercise 2 — Metadata
object metadata에 다음 정보를 넣어 저장/복원하십시오.

```text
organism
batch
operator
```

### Exercise 3 — Provenance
DNA → RNA → Protein chain을 만든 뒤 각 object의 parent ID를 출력하십시오.

### Exercise 4 — Broken Schema
JSON payload의 `schema_version`을 `"99.0"`으로 바꾸고 load해 보십시오.

### Exercise 5 — Nested Schema
`Sample` payload 내부의 `FeatureSet` schema만 잘못된 version으로 바꾸어
왜 load가 실패해야 하는지 설명하십시오.

### Exercise 6 — ModelArtifact
새 `ModelArtifact`에 model name, target name, metadata를 넣고 저장/복원하십시오.

### Exercise 7 — Prediction Consistency
artifact save/load 전후 prediction이 같은지 assert로 검증하십시오.

### Exercise 8 — Architecture Question
왜 model file과 feature-column information을 별도로 관리하지 않고
`ModelArtifact` 하나로 묶는 것이 더 안전한지 설명하십시오.

### Exercise 9 — Security
왜 인터넷에서 받은 `.pkl` 또는 `.joblib` file을 바로 load하면 안 되는지 설명하십시오.

### Exercise 10 — SDL Preview
Self-Driving Lab에서 experiment result의 provenance가 왜 중요한지
한 가지 구체적 workflow를 가정해 설명하십시오.